# Public systems on NegPairs-Perm, NegPairs-Fresh and NevIR (revision step 1)

**Runtime:** Runtime > Change runtime type > **A100 GPU** (comes with High-RAM). An **L4** also works (about twice as
slow). A T4 (16 GB) cannot hold the 7B and 8B judges in bfloat16: on a T4 the notebook skips the pointwise LLM judges
and monoT5-3B automatically (set `FORCE_LARGE = True` to override).
**Nothing to upload.** The `cacsa2` package is embedded in the notebook (cell 2) and is archived with the results.
**Files on Drive (`MyDrive/CACSA_v3/`):** the folder `data/negpairs_v3/` with `perm.jsonl` and `fresh.jsonl` (already there
from the main experiment). The archived main run `runs/np_v3_main/` is used, if present, for an anchor check only.
**Secrets:** `HF_TOKEN` optional (needed only for the gated Llama judge); `ANTHROPIC_API_KEY` only if `RUN_API_NEVIR = True`.

`Runtime > Run all`. The run folder `MyDrive/CACSA_v3/runs/public_systems/` is reused across sessions: finished systems
are skipped, failed ones are retried (their errors are kept in `failures.json`).

No library beyond Colab's own (transformers, sentence-transformers, datasets) is installed; GTE-ModernColBERT (PyLate
format) is scored by a replication of PyLate's encoding that was checked to give identical scores, so the environment
is not changed by package pins.

What is scored. Every system is used as released, through its own interface, as a function s(query, document):
on NegPairs the query is the answer option and the document is "context + question" (the formulation of the article's
scorers and of M0-native); the pointwise LLM judges see context, question and option as separate fields. On NevIR the
four query-document combinations of each instance are scored. The metrics are the article's (flip rates, SS, SS_perm_bal
with scenario-cluster bootstrap by clause order; paired accuracy with strict comparisons on all / nested / complement).

| Phase | Expected on an A100 |
|---|---|
| Drive, setup, data checks, NevIR | 3 min |
| late interaction (ColBERTv2, answerai-colbert-small, GTE-ModernColBERT, bge-m3 three heads) | 10 min |
| dense (4), SPLADE, cross-encoders (4), monoT5-base, monoT5-3B | 20 min |
| pointwise LLM judges (Phi-4-mini, OLMo-2-7B, Qwen3-8B, Llama-3.1-8B if the token has access) | 40 to 50 min |
| metrics, tables, zip | 5 min |

The NegPairs sets were generated by Claude Sonnet 5 and verified by Claude Haiku 4.5 and Qwen2.5-7B-Instruct; those
three models are therefore not used as judges on NegPairs. Qwen3-8B is a later generation of the second verifier's
family and is flagged as such in the output. The optional API judge (Claude Haiku 4.5) is run on NevIR only.

Precision. monoT5 and the LLM judges run in bfloat16 (their weights), but the two or few logits that decide a score are
computed in float32 from the state entering the output head, so that near-identical documents are not tied by the
rounding of bfloat16 logits; `RESCORE` lists the systems whose earlier bfloat16-logit scoring is kept under
`scores/old/` and compared in `tables/precision.md`.

At the end `public_systems_results.zip` is downloaded (scores of every system, metrics, tables, environment).


In [ ]:
# ============================== CONFIGURATION ==============================
PROJECT_ROOT  = "/content/drive/MyDrive/CACSA_v3"
PACKAGE_SHA256 = "5479f82ae27baeab0dfe9b60d4f49610161b0c806cbd633f32afb153cdffbf9a"            # the cacsa2 package embedded in the next cell (nothing to upload)
DATA_V3_DIR   = f"{PROJECT_ROOT}/data/negpairs_v3"
DATA_V3_SHA256 = {
 "perm.jsonl": "0a1673da78a13ef1f3728d4d3863bfb1980bdeae0cdb6841b0e774ed00b88de3",
 "fresh.jsonl": "83944035b91f5521671c88b122a4ba5b60bf96fc427d1c654847211eb84b7596"
}
RUN_NAME      = "public_systems"
MAIN_RUN      = f"{PROJECT_ROOT}/runs/np_v3_main"       # archived main run (anchor check of ColBERTv2 native scores; optional)
NEVIR         = {"dataset": "orionweller/NevIR", "split": "test", "revision": "6263585072ce3b435ed09658613553fbf4e74184"}
NEVIR_COUNTS  = {"instances": 1383, "nested": 184, "equal_bags": 36}
N_BOOT        = 10000
FORCE_LARGE   = False          # True: run the 7B/8B judges and monoT5-3B even on a GPU with less than 20 GB
SKIP, ONLY    = [], None       # names to skip; or a list of names to run exclusively
RESCORE       = ["monot5-base", "monot5-3b", "phi-4-mini-instruct", "olmo-2-7b-instruct", "qwen3-8b"]   # score again (previous files kept under scores/old/)
PURGE_CACHE   = True           # delete the hub cache of large checkpoints after scoring (disk)
RUN_API_NEVIR = False          # optional: Claude Haiku 4.5 as a pairwise judge on NevIR (about 2 to 3 USD; needs ANTHROPIC_API_KEY)
API_MODEL, API_PRICE, API_CAP_USD = "claude-haiku-4-5-20251001", (1.0, 5.0), 4.0
SYSTEMS = [
    # late interaction (MaxSim over token vectors)
    {"name": "colbertv2", "type": "colbert_checkpoint", "repo": "colbert-ir/colbertv2.0", "revision": "c1e84128e85ef755c096a95bdb06b47793b13acf",
     "label": "ColBERTv2 (native scoring)", "batch": 32},
    {"name": "answerai-colbert-small-v1", "type": "colbert_checkpoint", "repo": "answerdotai/answerai-colbert-small-v1", "label": "answerai-colbert-small-v1", "batch": 32},
    {"name": "gte-moderncolbert-v1", "type": "st_colbert", "repo": "lightonai/GTE-ModernColBERT-v1", "label": "GTE-ModernColBERT-v1 (PyLate format)", "batch": 32},
    {"name": "bge-m3", "type": "bgem3", "repo": "BAAI/bge-m3", "label": "bge-m3 (dense, sparse and colbert heads)", "batch": 16, "purge": True},
    # dense bi-encoders
    {"name": "bge-base-en-v1.5", "type": "dense", "repo": "BAAI/bge-base-en-v1.5", "revision": "a5beb1e3e68b9ab74eb54cfd186867f64f240e1a",
     "query_prefix": "Represent this sentence for searching relevant passages: ", "label": "bge-base-en-v1.5", "batch": 64},
    {"name": "e5-base-v2", "type": "dense", "repo": "intfloat/e5-base-v2", "query_prefix": "query: ", "doc_prefix": "passage: ", "label": "e5-base-v2", "batch": 64},
    {"name": "all-mpnet-base-v2", "type": "dense", "repo": "sentence-transformers/all-mpnet-base-v2", "revision": "e8c3b32edf5434bc2275fc9bab85f82640a19130",
     "label": "all-mpnet-base-v2", "batch": 64},
    {"name": "gte-modernbert-base", "type": "dense", "repo": "Alibaba-NLP/gte-modernbert-base", "label": "gte-modernbert-base", "batch": 64},
    # learned sparse
    {"name": "splade-cocondenser-ensembledistil", "type": "splade", "repo": "naver/splade-cocondenser-ensembledistil", "label": "SPLADE (cocondenser-ensembledistil)", "batch": 32},
    # cross-encoders and monoT5 rerankers
    {"name": "ms-marco-minilm-l6-v2", "type": "cross_encoder", "repo": "cross-encoder/ms-marco-MiniLM-L6-v2", "alt_repos": ["cross-encoder/ms-marco-MiniLM-L-6-v2"],
     "label": "ms-marco-MiniLM-L6-v2", "batch": 64},
    {"name": "bge-reranker-base", "type": "cross_encoder", "repo": "BAAI/bge-reranker-base", "label": "bge-reranker-base", "batch": 32},
    {"name": "bge-reranker-v2-m3", "type": "cross_encoder", "repo": "BAAI/bge-reranker-v2-m3", "label": "bge-reranker-v2-m3", "batch": 32, "purge": True},
    {"name": "mxbai-rerank-base-v1", "type": "cross_encoder", "repo": "mixedbread-ai/mxbai-rerank-base-v1", "label": "mxbai-rerank-base-v1", "batch": 32},
    {"name": "monot5-base", "type": "monot5", "repo": "castorini/monot5-base-msmarco-10k", "label": "monoT5-base", "batch": 32},
    {"name": "monot5-3b", "type": "monot5", "repo": "castorini/monot5-3b-msmarco-10k", "label": "monoT5-3B", "batch": 8, "purge": True, "large": True},
    # pointwise LLM judges (open weights; none took part in the construction of the NegPairs sets)
    {"name": "phi-4-mini-instruct", "type": "llm_pointwise", "repo": "microsoft/Phi-4-mini-instruct", "label": "Phi-4-mini-instruct (3.8B)", "batch": 8, "purge": True, "large": True,
     "trust_remote_code_fallback": True},
    {"name": "olmo-2-7b-instruct", "type": "llm_pointwise", "repo": "allenai/OLMo-2-1124-7B-Instruct", "label": "OLMo-2-1124-7B-Instruct", "batch": 8, "purge": True, "large": True},
    {"name": "qwen3-8b", "type": "llm_pointwise", "repo": "Qwen/Qwen3-8B", "chat_kwargs": {"enable_thinking": False}, "label": "Qwen3-8B (thinking off)", "batch": 8,
     "purge": True, "large": True, "note": "same developer as verifier 2 (Qwen2.5-7B-Instruct); later generation"},
    {"name": "llama-3.1-8b-instruct", "type": "llm_pointwise", "repo": "meta-llama/Llama-3.1-8B-Instruct", "label": "Llama-3.1-8B-Instruct", "batch": 8, "purge": True,
     "large": True, "gated": True},
]
# ===========================================================================


In [ ]:
# ============================== EMBEDDED PACKAGE (cacsa2 5479f82ae27b; generated, do not edit) ==============================
PACKAGE_B64 = """
...
"""


In [ ]:
from google.colab import drive
drive.mount('/content/drive')                      # first, so that the Drive prompt does not wait for the installs
import os, sys, json, zipfile, time, csv, shutil, importlib, hashlib, subprocess, traceback, gc, base64, io
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece', 'protobuf'], check=False)   # T5 and XLM-R tokenizers; nothing else is installed
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import torch
ROOT = Path(PROJECT_ROOT)
PKG_BYTES = base64.b64decode(''.join(PACKAGE_B64.split()))
ZIP_SHA = hashlib.sha256(PKG_BYTES).hexdigest()
assert ZIP_SHA == PACKAGE_SHA256, 'the embedded package does not match its recorded SHA-256'
PKG_DIR = Path('/content') / f'cacsa2_pkg_{ZIP_SHA[:12]}'
shutil.rmtree(PKG_DIR, ignore_errors=True)
with zipfile.ZipFile(io.BytesIO(PKG_BYTES)) as z:
    z.extractall(PKG_DIR)
for _m in [m for m in list(sys.modules) if m == 'cacsa2' or m.startswith('cacsa2.')]:
    del sys.modules[_m]
sys.path = [p for p in sys.path if 'cacsa2_pkg_' not in p]
PKG_ROOT = next(PKG_DIR.rglob('cacsa2/__init__.py')).parent.parent
sys.path.insert(0, str(PKG_ROOT))
importlib.invalidate_caches()
import cacsa2
assert tuple(int(x) for x in cacsa2.__version__.split('.')) >= (0, 3, 4), f'package {cacsa2.__version__}; upload cacsa2_project.zip 0.3.4 or later'
from cacsa2 import negpairs as npx, nevir, exact, provenance, public_systems as ps
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or None
    if HF_TOKEN:
        os.environ['HF_TOKEN'] = HF_TOKEN
except Exception:
    pass
RUN = ROOT / 'runs' / RUN_NAME
SCORES = RUN / 'scores'; SCORES.mkdir(parents=True, exist_ok=True)
if not (RUN / f'cacsa2_runtime_{ZIP_SHA[:12]}.zip').exists():
    (RUN / f'cacsa2_runtime_{ZIP_SHA[:12]}.zip').write_bytes(PKG_BYTES)          # the code used, archived with the results
GPU = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip() if shutil.which('nvidia-smi') else 'none'
GPU_GB = (torch.cuda.get_device_properties(0).total_memory / 2**30) if torch.cuda.is_available() else 0.0
def save_json(name, obj):
    (RUN / name).write_text(json.dumps(obj, indent=1, default=str))
env = provenance.run_environment({'run_name': RUN_NAME, 'cacsa2_version': cacsa2.__version__, 'package_zip_sha256': ZIP_SHA, 'gpu_gb': round(GPU_GB, 1),
                                  'packages_extra': {p: provenance._pkg_version(p) for p in ['sentence-transformers', 'torch', 'sentencepiece', 'anthropic']},
                                  'config': {'nevir': NEVIR, 'n_boot': N_BOOT, 'systems': SYSTEMS, 'skip': SKIP, 'only': ONLY, 'rescore': RESCORE, 'force_large': FORCE_LARGE,
                                             'run_api_nevir': RUN_API_NEVIR, 'api': {'model': API_MODEL, 'price': API_PRICE, 'cap_usd': API_CAP_USD}}})
ENVF = RUN / 'run_environment.json'
sessions = json.loads(ENVF.read_text()) if ENVF.exists() else []
sessions.append(env); ENVF.write_text(json.dumps(sessions, indent=1))
print('cacsa2', cacsa2.__version__, '| zip', ZIP_SHA[:12], '| GPU:', GPU, f'({GPU_GB:.0f} GB)', '| run folder', RUN, '| session', len(sessions))


In [ ]:
# ---- data: NegPairs-Perm and NegPairs-Fresh (hashed against the audited files), NevIR test with the nested subset ------
D3 = Path(DATA_V3_DIR)
for name, sha in DATA_V3_SHA256.items():
    assert provenance.sha256_file(D3 / name) == sha, f'{name} differs from the audited NegPairs v3.0 file'
TESTS = {'perm': npx.load_split(D3, 'perm'), 'fresh': npx.load_split(D3, 'fresh')}
BY_ID = {name: {r.sample_id: r for r in rows} for name, rows in TESTS.items()}
assert len(TESTS['perm']) == 1096 and len({r.passage_id for r in TESTS['perm']}) == 274, 'Perm: 274 scenarios x 4 arms expected'
assert len(TESTS['fresh']) == 960 and len({r.passage_id for r in TESTS['fresh']}) == 240, 'Fresh: 240 scenarios x 4 arms expected'
NEVIR_INST = nevir.load_test(NEVIR['dataset'], NEVIR['split'], revision=NEVIR['revision'])
NESTED = exact.nested_instances(NEVIR_INST)
NESTED_IDS = [i['id'] for i in NESTED]
EQUAL_IDS = [i['id'] for i in NESTED if i['contained'] == 'equal']
got = {'instances': len(NEVIR_INST), 'nested': len(NESTED), 'equal_bags': len(EQUAL_IDS)}
assert got == NEVIR_COUNTS, f'NevIR differs from the main run: {got} vs {NEVIR_COUNTS}'
COMBOS = nevir.four_combos(NEVIR_INST)
NP_ITEMS = {name: [(o, r.context, r.question) for r in rows for o in r.options] for name, rows in TESTS.items()}
save_json('data_audit.json', {'perm': {'items': len(TESTS['perm']), 'scenarios': 274}, 'fresh': {'items': len(TESTS['fresh']), 'scenarios': 240},
                              'nevir': {**got, 'revision': NEVIR['revision']}, 'files': DATA_V3_SHA256})
print(f"Perm {len(TESTS['perm'])} items | Fresh {len(TESTS['fresh'])} items | NevIR {len(NEVIR_INST)} instances, nested {len(NESTED)} (equal bags {len(EQUAL_IDS)}) | "
      f"{len(COMBOS)} NevIR pairs, {sum(len(v) for v in NP_ITEMS.values())} NegPairs pairs")


In [ ]:
# ---- scoring loop: one system at a time, resumable; failures recorded and retried in the next session --------------------
FAILF = RUN / 'failures.json'
FAIL = json.loads(FAILF.read_text()) if FAILF.exists() else {}
OLD = SCORES / 'old'; OLD.mkdir(exist_ok=True)
def run_system(spec):
    name = spec['name']
    out = SCORES / f'{name}.json'
    if out.exists() and name in RESCORE and not (OLD / f'{name}.json').exists():
        shutil.move(str(out), str(OLD / f'{name}.json'))      # previous scoring kept for the precision comparison
    if out.exists():
        return 'done'
    if name in SKIP or (ONLY and name not in ONLY):
        return 'skipped by configuration'
    if spec.get('large') and GPU_GB < 20 and not FORCE_LARGE:
        return f'skipped: {GPU_GB:.0f} GB GPU (set FORCE_LARGE = True to run it anyway)'
    if spec.get('gated') and not HF_TOKEN:
        return 'skipped: gated checkpoint and no HF_TOKEN'
    t0 = time.time()
    repos = [spec['repo']] + list(spec.get('alt_repos', []))
    last = None
    for repo in repos:
        s = {**spec, 'repo': repo}
        try:
            try:
                sc = ps.build_scorer(s, device='cuda' if torch.cuda.is_available() else 'cpu', hf_token=HF_TOKEN)
            except Exception as e:
                if spec.get('trust_remote_code_fallback') and not s.get('trust_remote_code'):
                    print(f'  {name}: native load failed ({repr(e)[:120]}); retrying with trust_remote_code=True', flush=True)
                    s = {**s, 'trust_remote_code': True}
                    sc = ps.build_scorer(s, device='cuda' if torch.cuda.is_available() else 'cpu', hf_token=HF_TOKEN)
                else:
                    raise
            rev = ps.resolve_revision(repo, spec.get('revision'))
            res = {'name': name, 'label': spec.get('label', name), 'kind': sc.kind, 'spec': s, 'resolved_revision': rev, 'settings': sc.settings,
                   'heads': sc.heads, 'note': spec.get('note'), 'negpairs': {}, 'nevir': {}}
            heads = sc.heads or [None]
            b = spec.get('batch', 32)
            for h in heads:
                key = h or 'score'
                res['negpairs'][key] = {}
                for sname, rows in TESTS.items():
                    vals = sc.score_negpairs(NP_ITEMS[sname], batch_size=b, head=h)
                    res['negpairs'][key][sname] = {r.sample_id: [vals[2 * k], vals[2 * k + 1]] for k, r in enumerate(rows)}
                vals = sc.score_nevir([(c[2], c[3]) for c in COMBOS], batch_size=b, head=h)
                res['nevir'][key] = nevir.to_score_dict(COMBOS, vals)
            res['truncated_docs'] = getattr(sc, 'truncated_docs', None)
            if spec['type'] == 'bgem3':                       # re-implemented heads: compare with the reference implementations on 40 pairs
                res['checks'] = ps.bgem3_selfcheck(sc, [(c[2], c[3]) for c in COMBOS[:20]] + [(o, ps.negpairs_document(c, q)) for o, c, q in NP_ITEMS['perm'][:20]])
                print(f'  bge-m3 self-check: {res["checks"]}', flush=True)
            res['seconds'] = round(time.time() - t0, 1)
            res['gpu'] = GPU
            sc.close(); del sc
            gc.collect(); torch.cuda.empty_cache()
            out.write_text(json.dumps(res))
            FAIL.pop(name, None); FAILF.write_text(json.dumps(FAIL, indent=1))
            if spec.get('purge') and PURGE_CACHE:
                ps.purge_hub_cache(repo)
            return f'scored in {res["seconds"]:.0f}s (revision {str(rev)[:12]})'
        except Exception as e:
            last = e
            tb = traceback.format_exc()
            FAIL[name] = {'repo': repo, 'error': repr(e)[:500], 'traceback': tb[-4000:], 'utc': datetime.now(timezone.utc).isoformat()}
            FAILF.write_text(json.dumps(FAIL, indent=1))
            gc.collect(); torch.cuda.empty_cache()
            if 'out of memory' in str(e).lower():
                break
    return f'FAILED: {repr(last)[:200]}'
STATUS = {}
for spec in SYSTEMS:
    msg = run_system(spec)
    STATUS[spec['name']] = msg
    print(f"{spec['name']:36s} {msg}", flush=True)
save_json('status.json', STATUS)
print(f"done: {sum(1 for v in STATUS.values() if v == 'done' or v.startswith('scored'))} of {len(SYSTEMS)} | failed: {[k for k, v in STATUS.items() if v.startswith('FAILED')]}")


In [ ]:
# ---- anchor: ColBERTv2 native scores of this run against the archived main run (if the folder is present) -------------
ANCHOR = None
old = Path(MAIN_RUN) / 'm0_native_scores.json'
new = SCORES / 'colbertv2.json'
if old.exists() and new.exists():
    o = json.loads(old.read_text()); n = json.loads(new.read_text())
    ANCHOR = {s: ps.anchor_against_archived(n['negpairs']['score'][s], o['negpairs'][s], n['nevir']['score'], o['nevir']) for s in ('perm', 'fresh')}
    ANCHOR['note'] = 'main run scored the E6 model in its session dtype (bfloat16 on the GPU); this run in float32'
    save_json('anchor_colbertv2.json', ANCHOR)
    print(json.dumps(ANCHOR, indent=1))
else:
    print('anchor check skipped (archived main run or colbertv2 scores missing)')


In [ ]:
# ---- optional: API judge on NevIR (pairwise, both orders; cost metered and capped; resumable) ---------------------------
if RUN_API_NEVIR:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anthropic'], check=True)
    import anthropic
    from google.colab import userdata
    client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))
    def call(model, system, user):
        for attempt in range(6):
            try:
                r = client.messages.create(model=model, max_tokens=4, temperature=0, system=system, messages=[{'role': 'user', 'content': user}])
                return ''.join(getattr(b, 'text', '') for b in r.content), r.usage.input_tokens, r.usage.output_tokens
            except Exception as e:
                if attempt == 5:
                    raise
                time.sleep(2 * (attempt + 1))
    name = f"api-{API_MODEL}"
    out = SCORES / f'{name}.json'
    done = json.loads(out.read_text()) if out.exists() else {'name': name, 'label': f'{API_MODEL} (pairwise judge, NevIR only)', 'kind': 'api_pairwise',
                                                               'spec': {'name': name, 'type': 'api_pairwise', 'model': API_MODEL}, 'nevir': {'score': {}}, 'negpairs': {}}
    if len(done['nevir']['score']) < len(NEVIR_INST):
        judge = ps.ClaudePairwiseNevIR(API_MODEL, call, API_PRICE, API_CAP_USD, name=name)
        done['settings'] = judge.settings
        judge_out = done['nevir']['score']                      # updated in place by score_instances
        def progress(k, usd):
            if k % 50 == 0:
                done['cost'] = {'usd': usd, 'calls': judge.calls}
                out.write_text(json.dumps(done)); print(f'{k} instances, {usd:.2f} USD', flush=True)
        try:
            judge.score_instances(NEVIR_INST, done=judge_out, on_progress=progress)
        finally:
            done['cost'] = {'usd': judge.usd, 'calls': judge.calls, 'in_tokens': judge.in_tokens, 'out_tokens': judge.out_tokens}
            out.write_text(json.dumps(done))
    print(f"{name}: {len(done['nevir']['score'])} of {len(NEVIR_INST)} instances, cost so far {done.get('cost', {}).get('usd', 0):.2f} USD")
else:
    print('API judge not run (RUN_API_NEVIR = False)')


In [ ]:
# ---- metrics (the article's functions), tables, environment, zip ---------------------------------------------------------
ROWS = []
for f in sorted(SCORES.glob('*.json')):
    res = json.loads(f.read_text())
    for key in sorted(res['nevir']):
        name = res['name'] if key == 'score' else f"{res['name']}/{key}"
        row = {'name': name, 'label': res.get('label', name) + ('' if key == 'score' else f' [{key}]'), 'kind': res['kind'], 'spec': res.get('spec'),
               'resolved_revision': res.get('resolved_revision'), 'settings': res.get('settings'), 'note': res.get('note'), 'seconds': res.get('seconds'),
               'truncated_docs': res.get('truncated_docs'), 'cost': res.get('cost')}
        if key in res.get('negpairs', {}):
            recs = {s: ps.negpairs_records(TESTS[s], res['negpairs'][key][s]) for s in TESTS}
            row['perm'] = ps.perm_metrics(recs['perm'], BY_ID['perm'], N_BOOT)
            row['fresh'] = ps.fresh_metrics(recs['fresh'], BY_ID['fresh'], N_BOOT)
            row['perm_predictions'] = {r['sample_id']: r['pred'] for r in recs['perm']}
            row['fresh_predictions'] = {r['sample_id']: r['pred'] for r in recs['fresh']}
        sc = res['nevir'][key]
        if len(sc) == len(NEVIR_INST):
            row['nevir'] = ps.nevir_metrics(sc, NEVIR_INST, NESTED_IDS, EQUAL_IDS, N_BOOT)
        ROWS.append(row)
ORDER = {k: i for i, k in enumerate(['late_interaction', 'dense', 'sparse', 'cross_encoder', 'monot5', 'llm_pointwise', 'api_pairwise'])}
ROWS.sort(key=lambda r: (ORDER.get(r['kind'], 99), r['name']))
TAB = RUN / 'tables'; TAB.mkdir(exist_ok=True)
with_np = [r for r in ROWS if 'perm' in r]
with_nv = [r for r in ROWS if 'nevir' in r]
(TAB / 'perm.md').write_text(ps.perm_table(with_np))
(TAB / 'fresh.md').write_text(ps.fresh_table(with_np))
(TAB / 'nevir.md').write_text(ps.nevir_table(with_nv))
summary = {'systems': {r['name']: {'label': r['label'], 'kind': r['kind'], 'revision': r['resolved_revision'], 'note': r['note'], 'seconds': r['seconds'],
                                   'perm': {k: r['perm']['flat'][k] for k in ('all.SS_bal', 'all.pooled.SS', 'all.first.SS', 'all.second.SS', 'all.pooled.flip_O_N',
                                                                              'all.pooled.flip_O_S', 'all.pooled.flip_O_P', 'all.pooled.joint_NS_acc', 'all.pooled.acc_O',
                                                                              'all.pooled.acc_N', 'all.pooled.acc_S')} | {'SS_bal_ci': [r['perm']['SS_bal_all_ci']['low'], r['perm']['SS_bal_all_ci']['high']],
                                                                                                                         'ties': r['perm']['ties']} if 'perm' in r else None,
                                   'fresh': {'SS_all': r['fresh']['report']['all']['SS'], 'SS_seen': (r['fresh']['report']['seen'] or {}).get('SS'),
                                             'SS_unseen': (r['fresh']['report']['unseen'] or {}).get('SS'), 'flip_O_N': r['fresh']['report']['all']['flip_O_N'],
                                             'flip_O_S': r['fresh']['report']['all']['flip_O_S'], 'ties': r['fresh']['ties']} if 'fresh' in r else None,
                                   'nevir': {k: r['nevir'][k]['paired_accuracy'] for k in ('all', 'nested', 'complement')} | {'equal_bags': r['nevir'].get('equal_bags', {}).get('paired_accuracy'),
                                                                                                                              'per_query': r['nevir']['all']['per_query_accuracy'],
                                                                                                                              'instances_with_tie': r['nevir']['instances_with_tie']} if 'nevir' in r else None}
                       for r in ROWS},
           'failures': FAIL, 'status': STATUS, 'anchor': ANCHOR, 'gpu': GPU, 'n_boot': N_BOOT, 'data': json.loads((RUN / 'data_audit.json').read_text())}
PREC = {}
for f in sorted((SCORES / 'old').glob('*.json')):
    o = json.loads(f.read_text()); n = json.loads((SCORES / f.name).read_text()) if (SCORES / f.name).exists() else None
    if n is None:
        continue
    def block(res):
        b = {'head_precision': res.get('settings', {}).get('head_precision'), 'dtype': res.get('settings', {}).get('dtype')}
        sc = res['nevir']['score']
        b['nevir'] = ps.nevir_metrics(sc, NEVIR_INST, NESTED_IDS, EQUAL_IDS, 2000)
        b['nevir'] = {'paired_all': b['nevir']['all']['paired_accuracy'], 'paired_nested': b['nevir']['nested']['paired_accuracy'], 'instances_with_tie': b['nevir']['instances_with_tie']}
        for sname in TESTS:
            recs = ps.negpairs_records(TESTS[sname], res['negpairs']['score'][sname])
            m = ps.perm_metrics(recs, BY_ID['perm'], 2000) if sname == 'perm' else ps.fresh_metrics(recs, BY_ID['fresh'], 2000)
            b[sname] = {'SS': m['flat']['all.SS_bal'] if sname == 'perm' else m['report']['all']['SS'], 'ties': m['ties']}
        return b
    PREC[f.stem] = {'previous': block(o), 'rescored': block(n)}
summary['precision_check'] = PREC
if PREC:
    lines = ['| system | scoring | NevIR paired all | NevIR nested | NevIR ties | Perm SS_bal | Perm ties | Fresh SS | Fresh ties |', '|---|---|---|---|---|---|---|---|---|']
    for name, v in PREC.items():
        for key in ('previous', 'rescored'):
            b = v[key]
            lines.append(f"| {name} | {key}: {b['head_precision'] or b['dtype']} | {b['nevir']['paired_all']:.1f} | {b['nevir']['paired_nested']:.1f} | {b['nevir']['instances_with_tie']} | "
                         f"{b['perm']['SS']:.1f} | {b['perm']['ties']} | {b['fresh']['SS']:.1f} | {b['fresh']['ties']} |")
    (TAB / 'precision.md').write_text('\n'.join(lines)); print('\n'.join(lines)); print()
save_json('summary.json', summary)
save_json('metrics_full.json', [{k: v for k, v in r.items() if k not in ('perm_predictions', 'fresh_predictions')} for r in ROWS])
save_json('predictions.json', {r['name']: {'perm': r.get('perm_predictions'), 'fresh': r.get('fresh_predictions')} for r in ROWS if 'perm' in r})
print((TAB / 'perm.md').read_text()); print(); print((TAB / 'nevir.md').read_text())
n = provenance.write_file_manifest(RUN, RUN / 'FILE_MANIFEST_SHA256.csv')
RES_ZIP = ROOT / f'{RUN_NAME}_results.zip'
with zipfile.ZipFile(RES_ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(x for x in RUN.rglob('*') if x.is_file()):
        z.write(p, arcname=str(p.relative_to(RUN.parent)))
print(f'{n} files manifested | {RES_ZIP} ({RES_ZIP.stat().st_size // 1024 // 1024} MB)')
from google.colab import files
files.download(str(RES_ZIP))
